In [1]:
import torch 
import torchstain
import cv2
import numpy as np

# Immagine di riferimento
# Ho deciso di prendere una tile di Pannuke ben colorato
target = cv2.imread("/home/jovyan/path-vol/Flusso1/Decolorazione/PanNuke/Original_color/fold0/images/0_11.png")
target = cv2.cvtColor(target, cv2.COLOR_BGR2RGB)

# Inizializzazione del normalizzatore Macenko in backend torch
normalizer = torchstain.normalizers.MacenkoNormalizer(backend="torch")

# Tramite fit si fa una stima dei vettori di colorazione H&E sull'immagine target
# torchstain vuole un tensore (C,H,W) con valori [0,255]
target_tensor = torch.from_numpy(target).permute(2,0,1).float()
normalizer.fit(target_tensor)

In [6]:
# Funzione di normalizzazione in un singolo tile
def normalize_tile(img_path, normalizer):
    img = cv2.imread(img_path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img_tensor = torch.from_numpy(img).permute(2,0,1).float()

    try:
        norm_img, H, E = normalizer.normalize(I=img_tensor, stains=True)
        norm_img = norm_img.numpy().astype(np.uint8)
        return norm_img
    except Exception as e:
        # Gestione di tail problematici (in situazioni in cui quasi tutto è sfondo bianco)
        print(f"Normalizzazione fallita per {img_path}: {e}")
        return img.astype(np.uint8)

In [7]:
import os
from glob import glob
from tqdm import tqdm

def process_dataset(input_dir, output_dir, normalizer):
    os.makedirs(output_dir, exist_ok=True)
    tile_paths = glob(os.path.join(input_dir, "*.png"))

    for tile_path in tqdm(tile_paths, desc=f"Normalizzazione {input_dir}"):
        norm_img = normalize_tile(tile_path, normalizer)
        out_path = os.path.join(output_dir, os.path.basename(tile_path))
        norm_img_bgr = cv2.cvtColor(norm_img, cv2.COLOR_RGB2BGR)
        cv2.imwrite(out_path, norm_img_bgr)

# Applicazione del normalizer per tutti i dataset
datasets = [
    "/home/jovyan/path-vol/Flusso1/Decolorazione/PanNuke/Original_color/fold0/images/",
    "/home/jovyan/path-vol/Flusso1/Decolorazione/PanNuke/Original_color/fold1/images/",
    "/home/jovyan/path-vol/Flusso1/Decolorazione/PanNuke/Original_color/fold2/images/",
    "/home/jovyan/path-vol/Flusso1/Decolorazione/Glysac/Test/Images_vecchio",
    "/home/jovyan/path-vol/Flusso1/Decolorazione/Glysac/Train/Images_v",
    "/home/jovyan/path-vol/Flusso1/Decolorazione/Consep/fold0/images_or",
    "/home/jovyan/path-vol/Flusso1/Decolorazione/Consep/fold1/images"
]

for folder in datasets:
    output_folder = folder.replace("/home/jovyan/path-vol/Flusso1/Decolorazione", "/home/jovyan/path-vol/Flusso1/Decolorazione/normalized")
    process_dataset(folder, output_folder, normalizer)

Normalizzazione /home/jovyan/path-vol/Flusso1/Decolorazione/PanNuke/Original_color/fold0/images/:  64%|██████▍   | 1694/2656 [01:12<00:42, 22.52it/s]

Normalizzazione fallita per /home/jovyan/path-vol/Flusso1/Decolorazione/PanNuke/Original_color/fold0/images/0_1723.png: kthvalue(): Expected reduction dim 0 to have non-zero size.


Normalizzazione /home/jovyan/path-vol/Flusso1/Decolorazione/PanNuke/Original_color/fold0/images/:  72%|███████▏  | 1920/2656 [01:21<00:23, 31.26it/s]

Normalizzazione fallita per /home/jovyan/path-vol/Flusso1/Decolorazione/PanNuke/Original_color/fold0/images/0_1939.png: kthvalue(): Expected reduction dim 0 to have non-zero size.
Normalizzazione fallita per /home/jovyan/path-vol/Flusso1/Decolorazione/PanNuke/Original_color/fold0/images/0_1940.png: kthvalue(): Expected reduction dim 0 to have non-zero size.


Normalizzazione /home/jovyan/path-vol/Flusso1/Decolorazione/PanNuke/Original_color/fold0/images/:  72%|███████▏  | 1924/2656 [01:21<00:26, 27.57it/s]

Normalizzazione fallita per /home/jovyan/path-vol/Flusso1/Decolorazione/PanNuke/Original_color/fold0/images/0_1946.png: kthvalue(): Expected reduction dim 0 to have non-zero size.


... (47 outputs left out)

Normalizzazione fallita per /home/jovyan/path-vol/Flusso1/Decolorazione/PanNuke/Original_color/fold2/images/2_1862.png: kthvalue(): Expected reduction dim 0 to have non-zero size.


Normalizzazione /home/jovyan/path-vol/Flusso1/Decolorazione/PanNuke/Original_color/fold2/images/:  74%|███████▎  | 2007/2722 [01:23<00:28, 25.30it/s]

Normalizzazione fallita per /home/jovyan/path-vol/Flusso1/Decolorazione/PanNuke/Original_color/fold2/images/2_2023.png: kthvalue(): Expected reduction dim 0 to have non-zero size.


Normalizzazione /home/jovyan/path-vol/Flusso1/Decolorazione/PanNuke/Original_color/fold2/images/: 100%|██████████| 2722/2722 [01:53<00:00, 23.90it/s]
Normalizzazione /home/jovyan/path-vol/Flusso1/Decolorazione/Glysac/Test/Images_vecchio: 100%|██████████| 25/25 [00:06<00:00,  4.12it/s]
Normalizzazione /home/jovyan/path-vol/Flusso1/Decolorazione/Glysac/Train/Images_v: 100%|██████████| 34/34 [00:08<00:00,  4.11it/s]
Normalizzazione /home/jovyan/path-vol/Flusso1/Decolorazione/Consep/fold0/images_or: 100%|██████████| 27/27 [00:06<00:00,  4.39it/s]
Normalizzazione /home/jovyan/path-vol/Flusso1/Decolorazione/Consep/fold1/images: 100%|██████████| 14/14 [00:03<00:00,  4.43it/s]
